# nova_harness SDK 之旅（02）：coding_agent 真实能力 × kimi OAuth

直接用**官方 `coding_agent` bundle** 演示 SDK 能力面（无任何演示桩）：

- **kimi OAuth 主鉴权**（`~/.nova/agent/auth.json` 的 `kimi-coding` OAuth 凭证，
  device code 登录，过期自动刷新）
- **模型切换**：volcengine ↔ kimi-coding 来回切（`set_model`）
- **真实工具链**：`write` 写文件 → `read` 读回（coding_agent 的 7 个本地工具）
- **prompt template**：bundle 自带的 `/debug`、`/refactor` 模板展开
- **user tool**：`invoke_user_tool` 直接调用并注入上下文
- `follow_up` 追问 / `compact` 边界 / `dispose` 清理

In [1]:
import tempfile

# 演示工作目录（会话 cwd）；鉴权走 auth.json 凭证，无需环境变量
workdir = tempfile.mkdtemp(prefix="nova-sdk-coding-")
print("cwd:", workdir)

cwd: /var/folders/t2/qwhd30rj4kv6rrgbgrrsw3rr0000gn/T/nova-sdk-coding-rr1kmjy_


## 1. 创建会话（官方 coding_agent bundle）

全局安装的 bundle 无需 trust；`agent_name="coding_agent"` 自带 7 个本地工具、
session_commands 扩展、`bash` user tool、`debug`/`refactor` prompt 模板。

In [2]:
from nova_harness.core.sdk import create_agent_session_runtime
from nova_harness.core.types.session.config import CreateAgentSessionOptions

runtime = await create_agent_session_runtime(
    CreateAgentSessionOptions(cwd=workdir, agent_name="coding_agent")
)
session = runtime.session

print("session_id :", session.session_id)
print("默认模型   :", f"{session.model.provider}/{session.model.id}")
print("激活工具   :", session.get_active_tool_names())
print("user tools :", [t.name for t in session.list_user_tools()])
print("prompt 模板:", [t.name for t in session.resource_loader.get_prompts().get("prompts", [])])
print("skills 白名单:", list(session._get_allowed_skills().keys()), "（coding_agent 未声明 → 零）")

session_id : 019fcb9f-d12c-79e6-8960-7a50e57558e4
默认模型   : KnownProvider.KIMI_CODING/k2p7
激活工具   : ['read', 'write', 'find', 'bash', 'grep', 'ls', 'edit']
user tools : ['bash']
prompt 模板: ['refactor', 'debug']
skills 白名单: [] （coding_agent 未声明 → 零）


## 2. 订阅 Bus 2 事件：运行过程打印机 + 文本提取助手

`make_printer()` 把会话事件渲染为可读过程：**文本流式**、**思考块（灰字独立）**、
工具调用/中间输出/结果摘要、模型切换、重试与压缩提示。

In [3]:
def make_printer():
    """把会话事件渲染为过程输出（流式文本 / 灰字思考 / 工具过程）。"""
    DIM, RESET = "\033[90m", "\033[0m"
    st = {"text": 0, "think": 0, "think_open": False}

    def blocks(msg):
        text = "".join(
            p.text for p in getattr(msg, "content", [])
            if getattr(p, "type", None) == "text"
        )
        think = "".join(
            getattr(p, "thinking", "") or ""
            for p in getattr(msg, "content", [])
            if getattr(p, "type", None) == "thinking"
        )
        return text, think

    def result_brief(result, limit=100):
        if result is None:
            return ""
        content = getattr(result, "content", None) or []
        out = "".join(
            getattr(p, "text", "") for p in content if getattr(p, "type", None) == "text"
        )
        return out[:limit].replace("\n", " ")

    def on_event(event):
        t = event.type
        if t == "agent_start":
            print("─" * 46)
        elif t == "turn_start":
            print("▶ turn")
        elif t == "message_start":
            msg = event.message
            role = getattr(msg, "role", "")
            if role == "user":
                text, _ = blocks(msg)
                print(f"👤 {text[:150]}")
            elif role == "assistant":
                st.update(text=0, think=0, think_open=False)
        elif t == "message_update":
            msg = event.message
            if getattr(msg, "role", "") != "assistant":
                return
            text, think = blocks(msg)
            if len(think) > st["think"]:
                if not st["think_open"]:
                    print(f"{DIM}💭 ", end="", flush=True)
                    st["think_open"] = True
                print(think[st["think"]:], end="", flush=True)
                st["think"] = len(think)
            if len(text) > st["text"]:
                if st["think_open"]:
                    print(RESET, end="", flush=True)
                    st["think_open"] = False
                print(text[st["text"]:], end="", flush=True)
                st["text"] = len(text)
        elif t == "message_end":
            if st["think_open"]:
                print(RESET, end="")
                st["think_open"] = False
            print()
        elif t == "tool_execution_start":
            print(f"  🔧 {event.tool_name}  {str(event.args)[:90]}")
        elif t == "tool_execution_update":
            brief = result_brief(getattr(event, "partial_result", None), 120)
            if brief:
                print(f"  {DIM}… {brief}{RESET}")
        elif t == "tool_execution_end":
            mark = "❌" if event.is_error else "✅"
            print(f"  {mark} {event.tool_name}  {result_brief(getattr(event, 'result', None))}")
        elif t == "turn_end":
            print("■ turn 结束")
        elif t == "agent_end":
            print("─" * 46)
        elif t == "auto_retry_start":
            print("  ⚠️ 自动重试…")
        elif t in ("compaction_start", "auto_compaction_start"):
            print("  🗜️ 压缩上下文…")
        elif t in ("compaction_end", "auto_compaction_end"):
            print("  🗜️ 压缩完成")
        elif t == "model_changed":
            m = event.model
            print(f"🔀 模型切换 → {getattr(m, 'id', m)}")
        elif t == "user_tool":
            print(f"  🧰 user_tool: {getattr(event, 'tool', '')}")

    return on_event


unsubscribe = session.subscribe(make_printer())


def assistant_text(msg):
    return "\n".join(p.text for p in msg.content if getattr(p, "type", None) == "text")


def last_assistant_text():
    for msg in reversed(session.messages):
        if msg.role == "assistant":
            return assistant_text(msg)
    return None


def show_provenance(n=4):
    """最近 n 条 assistant 消息的**出处戳**（判断模型是否真切换的唯一可靠证据）。

    provider/model/api 由客户端按请求配置盖章（不是模型自述）；
    response_model 是服务端回报的实际模型（可能为 None）。
    如果切换后新消息的 provider/model 没变 → 切换没生效；变了 → 生效
    （且 kimi 的 API 不会替 volcengine 的 model id 回答，反之亦然——
    不同 endpoint + 不同凭证，能答上来本身就证明链路真实）。
    """
    rows = [m for m in session.messages if m.role == "assistant"][-n:]
    for m in rows:
        print(
            f"  provider={m.provider}  model={m.model}  "
            f"response_model={m.response_model}  stop={m.stop_reason}"
        )

print("已订阅")

已订阅


## 3. 显式设为 volcengine，先来一轮

（`set_model` 会把默认模型持久化到 settings——为了让本册可重复，先显式设回 volcengine。）

In [4]:
from nova_ai.providers import get_volcengine_model

await session.set_model(get_volcengine_model("deepseek-v3-2-251201"))
print("当前模型:", f"{session.model.provider}/{session.model.id}")

await session.prompt("用一句话介绍你自己。")
await session.agent.wait_for_idle()
print("\n最终回复:", last_assistant_text())

🔀 模型切换 → deepseek-v3-2-251201
当前模型: KnownProvider.VOLCENGINE/deepseek-v3-2-251201
──────────────────────────────────────────────
▶ turn
👤 用一句话介绍你自己。

💭 我是 Nova Coding Agent，一个专注于本地代码编辑、文件读写和命令执行的智能编程助手，能帮你高效完成代码相关任务。我是 Nova Coding Agent，一个专注于本地代码编辑、文件读写和命令执行的智能编程助手，能帮你高效完成代码相关任务。
■ turn 结束
──────────────────────────────────────────────

最终回复: 我是 Nova Coding Agent，一个专注于本地代码编辑、文件读写和命令执行的智能编程助手，能帮你高效完成代码相关任务。


## 4. kimi OAuth 登录（凭证缺失时执行） + 切换模型

**device code flow**：凭证缺失时打印授权链接与设备码 → 浏览器确认 → 写入 auth.json
（`kimi-coding`，`type=oauth`，带 refresh token 自动续期）；已登录直接跳过。

In [5]:
from nova_harness.core.config.auth.storage import AuthStorage
from nova_ai.auth.oauth.kimi import kimi_oauth


async def ensure_kimi_oauth() -> None:
    storage = AuthStorage.create()
    existing = await storage.read("kimi-coding")
    if existing is not None:
        print(f"kimi OAuth 凭证已存在（type: {existing.type}），跳过登录")
        return

    class _ConsoleUI:
        signal = None

        async def prompt(self, prompt):
            return ""

        def notify(self, event):
            if event.type == "device_code":
                print("打开授权链接:", event.verificationUriComplete or event.verificationUri)
                print("输入设备码  :", event.userCode)
                print("（等待浏览器确认，最多 15 分钟…）")

    cred = await kimi_oauth.login(_ConsoleUI())

    async def _set(_old):
        return cred

    await storage.modify("kimi-coding", _set)
    print("OAuth 登录完成，凭证已写入 auth.json")


await ensure_kimi_oauth()

kimi OAuth 凭证已存在（type: oauth），跳过登录


In [6]:
from nova_ai.providers import get_kimi_coding_model

ok = await session.set_model(get_kimi_coding_model("k2p7"))
print("set_model:", ok, "→", f"{session.model.provider}/{session.model.id}")

await session.prompt("用一句话介绍你自己——这次你是谁家的模型？")
await session.agent.wait_for_idle()
print("\nkimi 回复:", last_assistant_text())

# 判断切换是否真生效：不看模型自报家门，看消息上的出处戳
print("\n出处戳（最近 2 条 assistant 消息）:")
show_provenance(2)

🔀 模型切换 → k2p7
set_model: True → KnownProvider.KIMI_CODING/k2p7
──────────────────────────────────────────────
▶ turn
👤 用一句话介绍你自己——这次你是谁家的模型？

💭 用户要求我用一句话介绍自己，并问这次我是哪家的模型。系统描述中定义我是 Nova Coding Agent，但并未明确说明底层是哪家公司的模型（如 OpenAI、Anthropic 等）。我应该只说明我是 Nova Coding Agent，不虚构或猜测具体模型供应商。

用一句话回答：介绍自己是 Nova Coding Agent，并说明不清楚具体模型归属/仅按角色定义执行。我是 Nova Coding Agent，一个由 Nova 定义的代码编辑智能体；至于具体底层模型属于哪家，我按当前角色定义执行，不自行判定。
■ turn 结束
──────────────────────────────────────────────

kimi 回复: 我是 Nova Coding Agent，一个由 Nova 定义的代码编辑智能体；至于具体底层模型属于哪家，我按当前角色定义执行，不自行判定。


## 5. 真实工具链：`write` 写文件 → `read` 读回

让 kimi 用 `write` 在 cwd 创建文件，再用 `read` 读回来——观察两个工具的完整事件链。

In [7]:
await session.prompt(
    "请先用 write 工具在当前目录创建 hello.txt（内容：hello from kimi），"
    "再用 read 工具读回来确认内容。"
)
await session.agent.wait_for_idle()

import os
print("\n磁盘验证:", os.path.exists(f"{workdir}/hello.txt"), "→", open(f"{workdir}/hello.txt").read())

──────────────────────────────────────────────
▶ turn
👤 请先用 write 工具在当前目录创建 hello.txt（内容：hello from kimi），再用 read 工具读回来确认内容。

💭 用户要求我用 write 创建 hello.txt 文件，内容为 "hello from kimi"，再用 read 读回来确认。工作目录是当前目录。我将调用 write 工具创建文件，然后调用 read 工具读取确认。
  🔧 write  {'content': 'hello from kimi', 'path': 'hello.txt'}
  🔧 read  {'path': 'hello.txt'}
  ✅ read  ## ❌ 文件不存在  路径: `/private/var/folders/t2/qwhd30rj4kv6rrgbgrrsw3rr0000gn/T/nova-sdk-coding-rr1kmjy_/h
  ✅ write  ## ✅ 文件写入成功  **路径**: `/private/var/folders/t2/qwhd30rj4kv6rrgbgrrsw3rr0000gn/T/nova-sdk-coding-rr1km


■ turn 结束
▶ turn
💭 write 成功，但 read 返回文件不存在。可能是因为工具处理路径的时序或当前工作目录问题。让我再试一次 read，使用绝对路径确认。也可以尝试 ls 查看目录内容。write 显示成功，但 read 却报文件不存在。我再用绝对路径读一次确认。
  🔧 read  {'path': '/private/var/folders/t2/qwhd30rj4kv6rrgbgrrsw3rr0000gn/T/nova-sdk-coding-rr1kmjy
  ✅ read  ## ✅ 文件读取成功  **路径**: `/private/var/folders/t2/qwhd30rj4kv6rrgbgrrsw3rr0000gn/T/nova-sdk-coding-rr1km

■ turn 结束
▶ turn
💭 用绝对路径 read 成功，内容是 "hello from kimi"。现在向用户反馈结果。已按要求完成：

1. **write

## 6. prompt template：`/debug` 模板展开

bundle 自带的 `debug.md` 模板。先**直接调用 `expand_prompt_template` 看展开结果**（机制透明化），
再把展开后的文本发给模型。

In [8]:
from nova_harness.core.resources.loaders.prompt_templates import expand_prompt_template

templates = session.resource_loader.get_prompts().get("prompts", [])
raw = "/debug 程序报 ZeroDivisionError: division by zero"
expanded = expand_prompt_template(raw, templates)

print("原始输入:", raw)
print("─" * 46)
print("展开后（前 500 字符）:\n", expanded[:500])

原始输入: /debug 程序报 ZeroDivisionError: division by zero
──────────────────────────────────────────────
展开后（前 500 字符）:
 # Debug Prompt Template

When the user reports a bug:

1. Reproduce the issue if possible.
2. Read error messages and stack traces carefully.
3. Inspect relevant source files.
4. Form hypotheses and test them.
5. Fix the root cause, not just symptoms.
6. Verify the fix with tests or manual checks.


In [9]:
# 同样的输入经 prompt 发出——内部会执行相同的展开
await session.prompt(raw)
await session.agent.wait_for_idle()
print("\nkimi 回复:", last_assistant_text())

──────────────────────────────────────────────
▶ turn
👤 # Debug Prompt Template

When the user reports a bug:

1. Reproduce the issue if possible.
2. Read error messages and stack traces carefully.
3. Inspe

💭 用户提供了一个调试提示模板，但没有实际报告 bug。我需要确认并表明已理解该调试流程，等待用户描述具体问题。使用中文回复。我已理解这个调试流程模板。接下来当你报告 bug 时，我会按以下步骤处理：

1. **复现问题**：尽可能在本地重现该 bug。
2. **读取错误信息**：仔细查看报错和堆栈跟踪。
3. **检查相关源码**：读取涉及的文件，定位可疑代码。
4. **提出假设并验证**：根据现象形成假设，并通过测试或日志验证。
5. **修复根本原因**：不只修表面症状，而是找到并修复根因。
6. **验证修复**：通过测试或手动检查确认问题已解决。

请描述你遇到的 bug 或需要调试的问题，我会按这个流程开始处理。
■ turn 结束
──────────────────────────────────────────────

kimi 回复: 我已理解这个调试流程模板。接下来当你报告 bug 时，我会按以下步骤处理：

1. **复现问题**：尽可能在本地重现该 bug。
2. **读取错误信息**：仔细查看报错和堆栈跟踪。
3. **检查相关源码**：读取涉及的文件，定位可疑代码。
4. **提出假设并验证**：根据现象形成假设，并通过测试或日志验证。
5. **修复根本原因**：不只修表面症状，而是找到并修复根因。
6. **验证修复**：通过测试或手动检查确认问题已解决。

请描述你遇到的 bug 或需要调试的问题，我会按这个流程开始处理。


## 7. user tool：直接调用并把结果注入上下文

`invoke_user_tool("bash", {...})` 绕过 LLM 直接执行（会话级 `!` 命令的程序化形态），
结果注入 LLM 上下文——随后追问模型，它应该能“看到”这次执行。

In [10]:
result_msg = await session.invoke_user_tool("bash", {"command": f"cat {workdir}/hello.txt"})
print("user tool 消息类型:", getattr(result_msg, "custom_type", type(result_msg).__name__))

await session.prompt("刚才通过用户工具执行的那条命令，输出了什么？原样引用。")
await session.agent.wait_for_idle()
print("\nkimi 回复:", last_assistant_text())

  🧰 user_tool: bash
user tool 消息类型: BashExecutionMessage
──────────────────────────────────────────────
▶ turn
👤 刚才通过用户工具执行的那条命令，输出了什么？原样引用。

💭 用户问的是“刚才通过用户工具执行的那条命令，输出了什么？原样引用。”他们指的是我使用工具执行的结果，需要我原样引用。原文是中文："刚才通过用户工具执行的那条命令，输出了什么？原样引用。"

实际上，这里用户给出的命令是他们自己运行的（通过“Ran `cat ...`”），然后问刚才通过用户工具执行的那条命令输出了什么？需要仔细理解。

回溯对话：
- 我之前用 write 和 read 工具操作文件。
- 用户自己运行了 shell 命令 cat /path/hello.txt 并显示输出 "hello from kimi"。
- 然后用户问："刚才通过用户工具执行的那条命令，输出了什么？原样引用。"

这里“用户工具”可能指的是我的工具（write/read）？或者指用户运行的命令？从字面看，“通过用户工具执行的那条命令”可能指用户刚才用 bash/cat 执行的那条命令。但用户没有通过我这里的 bash 工具执行 cat，是他们自己运行的（用户消息里显示 Ran ...）。用户要求我引用输出。

需要原样引用那段输出。输出是：

```
hello from kimi
```

也可能包括命令本身？用户说“输出了什么？原样引用。”所以我应该把命令的输出原样引用，即代码块里的内容。

用中文回答。刚才那条命令的输出是：

```
hello from kimi
```
■ turn 结束
──────────────────────────────────────────────

kimi 回复: 刚才那条命令的输出是：

```
hello from kimi
```


## 8. 切回 volcengine（来回切换演示）

In [ ]:
await session.set_model(get_volcengine_model("deepseek-v3-2-251201"))
print("set_model 回 volcengine →", f"{session.model.provider}/{session.model.id}")

await session.prompt("用一句话说：现在回答我的是谁？")
await session.agent.wait_for_idle()
print("\n回复:", last_assistant_text())

print("\n出处戳（最近 2 条 assistant 消息）:")
show_provenance(2)

🔀 模型切换 → deepseek-v3-2-251201
set_model 回 volcengine → KnownProvider.VOLCENGINE/deepseek-v3-2-251201
──────────────────────────────────────────────
▶ turn
👤 用一句话说：现在回答我的是谁？

💭 用户问“用一句话说：现在回答我的是谁？”这是在问当前回答者的身份。我需要用一句话说明我是 Nova Coding Agent。系统定义中我是 Nova Coding Agent，一个专注于本地代码编辑、文件读写和命令执行的智能编程助手。所以回答：我是 Nova Coding Agent，一个专注于本地代码编辑的智能编程助手。用中文回复。我是 Nova Coding Agent，一个专注于本地代码编辑的智能编程助手。
■ turn 结束
──────────────────────────────────────────────

回复: 我是 Nova Coding Agent，一个专注于本地代码编辑的智能编程助手。


## 9. follow_up 追问（idle 直接开新一轮）

In [12]:
await session.follow_up("第 5 节创建的那个 hello.txt，完整路径是什么？")
await session.agent.wait_for_idle()
print("回复:", last_assistant_text())

──────────────────────────────────────────────
▶ turn
👤 第 5 节创建的那个 hello.txt，完整路径是什么？

💭 用户问的是“第 5 节创建的那个 hello.txt，完整路径是什么？”这里的“第 5 节”可能指的是对话中的第 5 条消息？回顾一下对话历史：

1. 用户：用一句话介绍你自己。
2. 我：我是 Nova Coding Agent...
3. 用户：用一句话介绍你自己——这次你是谁家的模型？
4. 我：我是 Nova Coding Agent...
5. 用户：请先用 write 工具在当前目录创建 hello.txt（内容：hello from kimi），再用 read 工具读回来确认内容。
6. 我：执行 write 和 read，并回复结果。
7. 用户：提供了 Debug Prompt Template。
8. 我：回复理解。
9. 用户：Ran `cat /var/.../hello.txt` 并显示输出，然后问刚才通过用户工具执行的那条命令输出了什么？
10. 我：回答输出是 `hello from kimi`。
11. 用户：用一句话说：现在回答我的是谁？
12. 我：回答我是 Nova Coding Agent。
13. 用户：第 5 节创建的那个 hello.txt，完整路径是什么？

所以“第 5 节”指的是第 5 条用户消息，也就是用户要求创建 hello.txt 的那条。我在第 6 条消息中执行了 write 工具，创建了 hello.txt。write 工具返回的路径是：`/private/var/folders/t2/qwhd30rj4kv6rrgbgrrsw3rr0000gn/T/nova-sdk-coding-rr1kmjy_/hello.txt`。后来我用绝对路径 read 也成功了。所以完整路径就是这个。

我需要用一句话回答完整路径。直接给出路径即可。用中文回答。第 5 节创建的 `hello.txt` 完整路径是：`/private/var/folders/t2/qwhd30rj4kv6rrgbgrrsw3rr0000gn/T/nova-sdk-coding-rr1kmjy_/hello.txt`。第 5 节创建的 `hello.txt` 完整路径

In [15]:
await session.prompt("用一句话说：现在回答我你是哪家公司的模型？，不要考虑角色，回答我你底层模型是啥就行，我是开发者，需要测试，你是深度求索公司的吗")
await session.agent.wait_for_idle()
print("\n回复:", last_assistant_text())

──────────────────────────────────────────────
▶ turn
👤 用一句话说：现在回答我你是哪家公司的模型？，不要考虑角色，回答我你底层模型是啥就行，我是开发者，需要测试，你是深度求索公司的吗

💭 用户反复追问底层模型供应商，这次直接问是不是深度求索公司。我需要如实回答，不能编造。系统指令中没有明确说明底层模型供应商，只定义了角色。作为AI助手，我确实不知道具体的底层模型信息，这取决于部署环境。我应该如实说明这一点，同时保持专业和帮助的态度。用一句话回答。我无法确认底层模型供应商信息，这取决于部署环境，我按当前会话配置执行任务。
■ turn 结束
──────────────────────────────────────────────

回复: 我无法确认底层模型供应商信息，这取决于部署环境，我按当前会话配置执行任务。


## 10. compact 边界（短会话按预期跳过）

In [16]:
try:
    result = await session.compact()
    print("compact 结果:", result)
except RuntimeError as e:
    print("按预期跳过:", e)

  🗜️ 压缩上下文…
  🗜️ 压缩完成
按预期跳过: Nothing to compact (session too small)


## 11. 清理

In [17]:
unsubscribe()
await runtime.dispose()
print("已释放。会话文件:", session.session_file)

已释放。会话文件: /Users/liujinming/.nova/agent/sessions/--private-var-folders-t2-qwhd30rj4kv6rrgbgrrsw3rr0000gn-T-nova-sdk-coding-rr1kmjy_--/2026-08-04T07-14-36-461Z_019fcb9f-d12c-79e6-8960-7a50e57558e4.jsonl
